# Positional Encoding — Senoidal vs Aprendida

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


## Intuição

Self-attention é invariante a permutação: embaralhe os tokens e a saída embaralha junto. Para dar noção de ordem, *somamos* um vetor de posição ao embedding de cada token. Encodings senoidais têm propriedades elegantes (extrapolação, deslocamento relativo), mas posições aprendidas podem ser mais flexíveis dentro do range de treino.


## Formulação Matemática

Senoidal:

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

Para qualquer deslocamento fixo $k$, $PE_{pos+k}$ é uma função linear de $PE_{pos}$, então o modelo consegue raciocinar por posição relativa.


## Implementação


In [ ]:
import math, torch
import torch.nn as nn
import matplotlib.pyplot as plt


In [ ]:
def sinusoidal(L, d):
    pe = torch.zeros(L, d)
    pos = torch.arange(L).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

class LearnedPE(nn.Module):
    def __init__(self, max_len, d):
        super().__init__()
        self.pe = nn.Embedding(max_len, d)
    def forward(self, x):
        idx = torch.arange(x.size(1), device=x.device)
        return x + self.pe(idx)


## Experimento


In [ ]:
pe = sinusoidal(L=128, d=64)
plt.figure(figsize=(8, 4))
plt.imshow(pe.numpy(), aspect='auto', cmap='RdBu')
plt.colorbar(label='value'); plt.xlabel('dim'); plt.ylabel('position')
plt.title('Sinusoidal positional encoding'); plt.show()


In [ ]:
# Check the relative-shift property: PE[pos+k] should be a linear function of PE[pos]
diff = torch.cdist(pe[:1], pe).squeeze()
plt.plot(diff.numpy())
plt.xlabel('position'); plt.ylabel('distance to PE[0]')
plt.title('Distance grows smoothly — supports relative reasoning'); plt.show()


## Discussão

- Senoidal: zero parâmetros, extrapola para sequências maiores que as vistas em treino, codifica deslocamentos *relativos* limpos.
- Aprendida: poucos parâmetros extras por posição, pega particularidades do dataset finito, mas não extrapola além do range treinado.
- Alternativas modernas incluem **rotary** (RoPE) e **ALiBi**, que injetam posição direto dentro da atenção, não no embedding.


## Referências

- Repositório da série: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Autor: [Nandobez](https://github.com/Nandobez)
